In [ ]:
import os, shutil
import pandas as pd, numpy as np
# sys.path.append('/home/m.jaraiz/repos/pyLowOrder/')

from FotR import GANDALF


# CODA 2025

In [ ]:
try:
    shutil.rmtree("./rans3_transonic/")
except:
    print('No la encuentro')
    pass
# import pandas as pd
# df_post = pd.read_csv('/home/m.jaraiz/repos/CETACEO_UPM/cetaceo/data/f22/rans3/metadata/df_post.csv', sep=',', usecols=['aoa', 'mach', 'h', 'case_idx'])

In [ ]:

# EAREN = EarendilsLight(FRODO)

# EAREN.help("CODAReader")

dataset_dir = os.getcwd()

gimli = GANDALF(
    os.path.join(dataset_dir, "rans3_transonic"),
    eq_type="rans", 
    num_stages = 2,
    version="flowsimulator2024"
)
# cases = [64, 79, 87, 88, 94]
# gimli.define_cases(
#     method="external",
#     external_dataframe = pd.DataFrame({ # Caso 79 de df_post.csv
#         "AoA": df_post['aoa'].loc[cases],
#         "Mach": df_post['mach'].loc[cases],
#         "h": df_post['h'].loc[cases]
#     }),
# )

gimli.define_cases(
    method="Halton",
    bounds={"AoA": (0.0, 5.0), "Mach": (0.8, 1.1)}, # {"AoA": (0.0, 5.0), "Mach": (0.3, 1.5), 'h': 11000},
    n_samples=20,
    peak_ranges=None,
    seed=42,
)


In [ ]:
geom_csv = 'eta_0_0.csv'

gimli.define_geom_file(
    geom_file_path=os.path.join(dataset_dir, 'airfoil_files', geom_csv), sep=',', decimal='.', header = 'infer',
    cols_idx=[1, 3],
    normalize = False
)

c = gimli.array_ptos[:, 0].max() - gimli.array_ptos[:, 0].min()
cm = -c/4 # la malla lo tiene ya ajustado a x=0 el borde de ataque


In [ ]:
print(gimli.df_cases.columns)
T, P, rho = GANDALF.Backpack.isa_atmosphere(h=11000)
mu = GANDALF.Backpack.Sutherland_law(mu0 = 1.716e-5, T=T, Treference=255.55)

gimli.compute_param(
    name = 'Re',
    formula = "L * Mach*sqrt(1.4*287*T)*rho/mu",
    externals={
        "rho": rho,
        "mu": mu,
        "L": c,
        "T": T
    }
)
print(gimli.df_cases.columns)

In [ ]:
gimli.generate_folders(
    base_files = ["run_sst_v4.py", "run.sh"],
    mesh_path=os.path.join(dataset_dir, "sources", "mesh_f22_transonic.msh"),
    script_dir=os.path.join(dataset_dir, "sources"),
    folder_fmt = "aoa_{AoA:.4f}_mach_{Mach:.4f}",
    overwrite=True,
    update_base_files=True,
    data_to_update={
        "AOA_PLACEHOLDER": "AoA",
        "MACH_PLACEHOLDER": "Mach",
        "ALT_PLACEHOLDER": "h",
        "PYTHON_FILE_PLACEHOLDER": "run_sst_v4.py",
        "CHORD_PLACEHOLDER": str(c),
        "CM_PLACEHOLDER": str(cm)
    }
)

In [ ]:
# df_sacar = gimli.df_cases.copy()
# df_sacar['AoA'] = pd.to_numeric(df_sacar['AoA'])
# df_sacar['Mach'] = pd.to_numeric(df_sacar['Mach'])
# df_definitivo = df_sacar.round({'AoA': 4, 'Mach': 4})
# df_definitivo['Coef_Area'] = c
# df_definitivo.to_csv(os.path.join(gimli.root_dir,'metadata', 'df_cases.csv'), index=False)
gimli.df_cases['Coef_Area'] = c
gimli.df_cases.to_csv(os.path.join(gimli.root_dir,'metadata', 'df_cases.csv'), index=False)

In [ ]:
gimli.assign_jobs(
    file_sh="run.sh",
    nodes=[f'n00{n}' for n in [3, 4, 5]],
    cpus_per_job=48,
    submit=True,
    )

In [ ]:
gimli.submit_cases()

# FLUENT R22

In [ ]:
try:
    shutil.rmtree("./f22_fluent/outputs")
except:
    pass

gimli = GIMLI("./f22_fluent/", eq_type="rans")

gimli.define_cases(
    method="Halton",
    bounds={"AoA": (0.0, 5.0), "Mach": (0.3, 1.5), 'h': 11000},
    n_samples=5,
    peak_ranges=None,
    seed=42,
)

dataset_dir = "/home/m.jaraiz/repos/CETACEO_UPM/cetaceo/data/f22/airfoil_files/"
geom_csv = 'eta_0_0.csv'

gimli.define_geom_file(
    geom_file_path=os.path.join(dataset_dir, geom_csv), sep=',', decimal='.', header = 'infer',
    cols_idx=[1, 3],
    normalize = False
)

c = gimli.array_ptos[:, 0].max() - gimli.array_ptos[:, 0].min()
T, P, rho = GIMLI.Backpack.isa_atmosphere(h=11000)
mu = GIMLI.Backpack.Sutherland_law(mu0 = 1.716e-5, T=T, Treference=255.55)

gimli.compute_param(
    name = 'Re',
    formula = "L * Mach*sqrt(1.4*287*T)*rho/mu",
    externals={
        "rho": rho,
        "mu": mu,
        "L": c,
        "T": T
    }
)

gimli.compute_param(
    name = 'T',
    formula = "T",
    externals={
        "T": T
    }
)

gimli.compute_param(
    name = 'P',
    formula = "P",
    externals={
        "P": P
    }
)

gimli.generate_folders(
    base_files=["journal.jou", "SLURM_FLUENT_2D.job"],
    script_dir="/home/m.jaraiz/repos/CETACEO_UPM/cetaceo/data/f22_fluent/sources/",
    folder_fmt = "aoa_{AoA:.4f}_mach_{Mach:.4f}_h_{h:.0f}",
    overwrite=True,
    update_base_files=True,
    data_to_update={
        "AOA_PLACEHOLDER": "AoA",
        "MACH_PLACEHOLDER": "Mach",
        "TEMP_PLACEHOLDER": "T",
        "P_PLACEHOLDER": "P",
        "FLUENT_TEST": "f22_fluent"
    }
    
)

# asignaciones = gimli.assign_jobs(
#     nodes =[f"n00{n}" for n in [5, 6, 7]],
#     cpus_per_job=48
# )

In [ ]:
gimli.df_cases

In [ ]:
n_cases = gimli.case_tensor.shape[0]
print(f"Creating {n_cases} simulation folders in {gimli.root_dir}")

for i, row in enumerate(gimli.case_tensor):
    
    case_dict = {var: float(row[j]) for j, var in enumerate(gimli.design_vars)}


In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.scatter(x=gimli.df_cases['AoA'], y= gimli.df_cases['Mach'], s=5, c=gimli.df_cases['Re'], cmap='turbo')
plt.colorbar()